# DataCo | Modeling Pipeline

**Modelo de Clasificación del Riesgo de Entrega Tardía**

## 1. Carga y preparación inicial de datos

En esta sección se importan las bibliotecas necesarias y se carga el conjunto de datos DataCo Smart Supply Chain.

El objetivo es establecer la base del flujo de preparación de datos para el modelo de clasificación, cuya variable objetivo es `Late_delivery_risk`.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Localizar el conjunto de datos
rutas_dataset = [
    Path("Dataset/DataCoSupplyChainDataset.csv"),
    Path("DataCoSupplyChainDataset.csv")
]

ruta_dataset = next(
    (ruta for ruta in rutas_dataset if ruta.exists()),
    None
)

if ruta_dataset is None:
    raise FileNotFoundError(
        "No se encontró DataCoSupplyChainDataset.csv. "
        "Colóquelo en la carpeta Dataset/ o junto al notebook."
    )

# Cargar el dataset
df = pd.read_csv(
    ruta_dataset,
    encoding="latin1",
    sep=";",
    low_memory=False
)

df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Name,Customer City,Customer Country,...,Order State,Order Status,Order Zipcode,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,Sporting Goods,Caguas,Puerto Rico,...,Java Occidental,COMPLETE,NaN,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,Sporting Goods,Caguas,Puerto Rico,...,Rajastï¿,PENDING,NaN,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,Sporting Goods,San Jose,EE. UU.,...,Rajastï¿,CLOSED,NaN,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,Sporting Goods,Los Angeles,EE. UU.,...,Queensland,COMPLETE,NaN,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,Sporting Goods,Caguas,Puerto Rico,...,Queensland,PENDING_PAYMENT,NaN,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [3]:
print(f"Registros: {df.shape[0]:,}")
print(f"Variables: {df.shape[1]}")

Registros: 180,519
Variables: 44


## 2. División estratificada de entrenamiento y prueba

Se divide el conjunto de datos en dos subconjuntos: 80 % para entrenamiento y 20 % para prueba.

Se utiliza un muestreo estratificado basado en la variable `Shipping Mode`, con el propósito de conservar la distribución de las modalidades de envío en ambos conjuntos.

Se establece `random_state=42` para garantizar la reproducibilidad de la división.


In [4]:
from sklearn.model_selection import train_test_split

# Definir variable objetivo
variable_objetivo = "Late_delivery_risk"

# Separar variables predictoras y variable objetivo
X = df.drop(columns=[variable_objetivo])
y = df[variable_objetivo]

# División estratificada
X_train_estratificado, X_test_estratificado, y_train_estratificado, y_test_estratificado = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=X["Shipping Mode"]
)

print(f"Entrenamiento: {len(X_train_estratificado):,} registros")
print(f"Prueba: {len(X_test_estratificado):,} registros")
print(f"Variable objetivo: {variable_objetivo}")

Entrenamiento: 144,415 registros
Prueba: 36,104 registros
Variable objetivo: Late_delivery_risk


## 3. Preparación del conjunto de entrenamiento

Se preparan los conjuntos de datos obtenidos mediante la división estratificada, manteniendo separadas las variables predictoras y la variable objetivo `Late_delivery_risk`.

El conjunto de entrenamiento se utilizará para ajustar los transformadores de preprocesamiento, mientras que el conjunto de prueba permanecerá reservado para evaluar posteriormente el modelo.

Las transformaciones se aplicarán únicamente a las características seleccionadas, excluyendo aquellas que puedan producir fuga de información.


In [5]:
# Preparar los conjuntos de entrenamiento y prueba
df_train_limpieza = X_train_estratificado.copy()
df_test_limpieza = X_test_estratificado.copy()

# Conservar las etiquetas por separado
df_train_labels = y_train_estratificado.copy()
df_test_labels = y_test_estratificado.copy()

print("Conjunto de entrenamiento:", df_train_limpieza.shape)
print("Conjunto de prueba:", df_test_limpieza.shape)
print("Etiquetas de entrenamiento:", df_train_labels.shape)
print("Etiquetas de prueba:", df_test_labels.shape)

Conjunto de entrenamiento: (144415, 43)
Conjunto de prueba: (36104, 43)
Etiquetas de entrenamiento: (144415,)
Etiquetas de prueba: (36104,)


## 4. Implementación de transformadores personalizados

Se definen transformadores personalizados para incorporar nuevas características al proceso de preparación de datos de DataCo.

Se implementan dos mecanismos de transformación:

- **ClusterSimilarity:** utiliza K-Means y funciones de base radial (RBF) para calcular la similitud de cada registro con los centroides de los clústeres identificados.
- **Transformación de ratios:** permite generar características derivadas mediante la relación entre dos atributos numéricos, como descuentos, beneficios e ingresos por unidad.

Estos transformadores se integrarán posteriormente en los pipelines de preprocesamiento para automatizar la ingeniería de características.


In [6]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans

class ClusterSimilarity(BaseEstimator, TransformerMixin):

    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(
            n_clusters=self.n_clusters,
            n_init=10,
            random_state=self.random_state
        )

        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        return rbf_kernel(
            X,
            self.kmeans_.cluster_centers_,
            gamma=self.gamma
        )

    def get_feature_names_out(self, names=None):
        return [
            f"Cluster {i} similarity"
            for i in range(self.n_clusters)
        ]
        
        
# Calcular ratio entre dos columnas
def column_ratio(X):
    numerador = X[:, [0]].astype(float)
    denominador = X[:, [1]].astype(float)

    return np.divide(
        numerador,
        denominador,
        out=np.full_like(numerador, np.nan),
        where=denominador != 0
    )

# Nombre de la columna generada
def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]

print("Transformadores personalizados definidos correctamente.")

Transformadores personalizados definidos correctamente.


## 5. Definición de pipelines de preprocesamiento

Se implementan pipelines de Scikit-learn para automatizar las transformaciones necesarias en la preparación de los datos de DataCo.

Los procesos contemplan:

- **Atributos numéricos:** imputación de valores faltantes mediante la mediana y estandarización con StandardScaler.
- **Ratios:** generación de características derivadas mediante divisiones entre atributos, seguida de imputación y estandarización.
- **Transformaciones logarítmicas:** aplicación de log1p para transformar atributos no negativos y posteriormente estandarizarlos.
- **Atributos categóricos:** imputación mediante la categoría más frecuente y codificación con OneHotEncoder.
- **Similitud geográfica:** estandarización de coordenadas y generación de características mediante ClusterSimilarity.

Estos pipelines serán integrados mediante ColumnTransformer para aplicar cada transformación a los atributos correspondientes.


In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    FunctionTransformer,
    StandardScaler,
    OneHotEncoder
)

# Pipeline para atributos numéricos
num_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

# Pipeline para generar ratios
def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(
            column_ratio,
            feature_names_out=ratio_name
        ),
        SimpleImputer(strategy="median"),
        StandardScaler()
    )

# Pipeline para transformaciones logarítmicas
log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(
        np.log1p,
        feature_names_out="one-to-one"
    ),
    StandardScaler()
)

# Pipeline para ratios con transformación logarítmica
def log_ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(
            column_ratio,
            feature_names_out=ratio_name
        ),
        SimpleImputer(strategy="median"),
        FunctionTransformer(
            np.log1p,
            feature_names_out="one-to-one"
        ),
        StandardScaler()
    )

# Pipeline para atributos categóricos
cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
)

# Pipeline para similitud geográfica
geo_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    ClusterSimilarity(
        n_clusters=10,
        gamma=1.0,
        random_state=42
    )
)

print("Pipelines de preprocesamiento definidos correctamente.")

Pipelines de preprocesamiento definidos correctamente.


## 6. Integración de transformaciones mediante ColumnTransformer

Se utiliza `ColumnTransformer` de Scikit-learn para integrar los pipelines de preprocesamiento y asignar las transformaciones correspondientes a cada grupo de atributos del conjunto de datos DataCo.

El proceso contempla el tratamiento de variables numéricas, la generación de ratios, las transformaciones logarítmicas, la codificación de atributos categóricos y la generación de características de similitud geográfica mediante K-Means.

Se conservan las variables originales seleccionadas y se incorporan las características derivadas. Los atributos no incluidos en las transformaciones se excluyen mediante `remainder="drop"`.

Esta configuración permite centralizar el preprocesamiento y aplicar las transformaciones de manera consistente durante las siguientes etapas del modelado.


In [8]:
from sklearn.compose import ColumnTransformer

# Variables numéricas originales
columnas_numericas = [
    "Days for shipment (scheduled)",
    "Order Item Quantity",
    "Sales",
    "Order Item Total",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Product Price",
    "Benefit per order",
    "Order Item Profit Ratio",
    "Latitude",
    "Longitude"
]

# Variables categóricas
columnas_categoricas = [
    "Type",
    "Customer Country",
    "Customer Segment",
    "Market",
    "Department Name",
    "Shipping Mode"
]

# Variables con transformación logarítmica
columnas_log = [
    "Sales",
    "Order Item Total",
    "Order Item Discount",
    "Product Price"
]

# Integrar los pipelines
preprocessing = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, columnas_numericas),

        ("discount_ratio", ratio_pipeline(),
         ["Order Item Discount", "Order Item Quantity"]),

        ("profit_ratio", ratio_pipeline(),
         ["Benefit per order", "Order Item Quantity"]),

        ("revenue_ratio", ratio_pipeline(),
         ["Order Item Total", "Order Item Quantity"]),

        ("log", log_pipeline, columnas_log),

        ("log_discount_ratio", log_ratio_pipeline(),
         ["Order Item Discount", "Order Item Quantity"]),

        ("log_revenue_ratio", log_ratio_pipeline(),
         ["Order Item Total", "Order Item Quantity"]),

        ("geo", geo_pipeline, ["Latitude", "Longitude"]),

        ("cat", cat_pipeline, columnas_categoricas)
    ],
    remainder="drop"
)

print("ColumnTransformer configurado correctamente.")
print("\nTransformaciones definidas:")

for nombre, transformador, columnas in preprocessing.transformers:
    print(f"- {nombre}: {columnas}")

ColumnTransformer configurado correctamente.

Transformaciones definidas:
- num: ['Days for shipment (scheduled)', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Item Discount', 'Order Item Discount Rate', 'Product Price', 'Benefit per order', 'Order Item Profit Ratio', 'Latitude', 'Longitude']
- discount_ratio: ['Order Item Discount', 'Order Item Quantity']
- profit_ratio: ['Benefit per order', 'Order Item Quantity']
- revenue_ratio: ['Order Item Total', 'Order Item Quantity']
- log: ['Sales', 'Order Item Total', 'Order Item Discount', 'Product Price']
- log_discount_ratio: ['Order Item Discount', 'Order Item Quantity']
- log_revenue_ratio: ['Order Item Total', 'Order Item Quantity']
- geo: ['Latitude', 'Longitude']
- cat: ['Type', 'Customer Country', 'Customer Segment', 'Market', 'Department Name', 'Shipping Mode']


## 7. Ejecución del preprocesamiento

Se ejecuta el flujo de preprocesamiento mediante `fit_transform()` sobre el conjunto de entrenamiento de DataCo.

El proceso aplica automáticamente las transformaciones configuradas en `ColumnTransformer`, incluyendo imputación, estandarización, generación de ratios, transformaciones logarítmicas, codificación categórica y similitud geográfica.

La matriz resultante se convierte en un DataFrame de pandas utilizando los nombres de las características generadas, con el propósito de facilitar su interpretación.

El resultado constituye el conjunto de características preparado para el entrenamiento posterior del modelo de clasificación de riesgo de entrega tardía.


In [9]:
# Ejecutar el preprocesamiento sobre los datos de entrenamiento
X_train_preparado = preprocessing.fit_transform(
    df_train_limpieza
)

# Obtener los nombres de las características generadas
nombres_caracteristicas = preprocessing.get_feature_names_out()

# Convertir la matriz procesada a DataFrame
df_train_procesado = pd.DataFrame(
    X_train_preparado,
    columns=nombres_caracteristicas,
    index=df_train_limpieza.index
)

# Mostrar información del resultado
print("Preprocesamiento completado correctamente.")
print("Tipo de matriz:", type(X_train_preparado))
print("Dimensiones originales:", df_train_limpieza.shape)
print("Dimensiones procesadas:", df_train_procesado.shape)
print("Cantidad de características:", len(nombres_caracteristicas))

# Visualizar las primeras filas
display(df_train_procesado.head(3))

Preprocesamiento completado correctamente.
Tipo de matriz: <class 'numpy.ndarray'>
Dimensiones originales: (144415, 43)
Dimensiones procesadas: (144415, 59)
Cantidad de características: 59


,num__Days for shipment (scheduled),num__Order Item Quantity,num__Sales,num__Order Item Total,num__Order Item Discount,num__Order Item Discount Rate,num__Product Price,num__Benefit per order,num__Order Item Profit Ratio,num__Latitude,...,cat__Department Name_Footwear,cat__Department Name_Golf,cat__Department Name_Health and Beauty,cat__Department Name_Outdoors,cat__Department Name_Pet Shop,cat__Department Name_Technology,cat__Shipping Mode_First Class,cat__Shipping Mode_Same Day,cat__Shipping Mode_Second Class,cat__Shipping Mode_Standard Class
32071,-0.677969,0.599535,-1.201388,-1.227771,-0.533503,1.399276,-0.904599,-0.767414,-3.676678,-1.170107,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
83230,0.777157,-0.775773,-0.557277,-0.453094,-0.885832,-1.301431,-0.078932,0.397092,0.790646,0.395909,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
164640,0.777157,1.287188,-0.026882,-0.074487,0.245835,0.404278,-0.653237,-0.085721,-0.085719,1.082728,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## 8. Entrenamiento y selección del algoritmo

En esta sección se prueban cuatro algoritmos distintos para ver cuál adivina mejor si un pedido llegará tarde:

- Descenso de gradiente estocástico (SGD)
- Regresión logística
- Árbol de decisión
- Bosque aleatorio (Random Forest)

Antes de aprender, cada algoritmo recibe los datos ya preparados automáticamente por `preprocessing`. Todos se usan con su configuración original, sin ajustes. Solo se fija una semilla (`random_state=42`) para que los resultados salgan iguales cada vez que se ejecuta el cuaderno.

Para evaluar, cada algoritmo se evalúa con validación cruzada de 5 particiones. En cada evaluación aprende con una parte de los datos y se le evalúa con otra que no ha visto. Al final se promedian las 5 notas.

Se usa una sola forma de calificar: el área bajo la curva ROC. Esta nota mide qué tan bien el algoritmo sabe ordenar los pedidos, del que tiene más riesgo de llegar tarde al que tiene menos. Se eligió porque en los datos más de la mitad de los pedidos llegan tarde (54.8 %). Con otras formas de calificar, un modelo que responde siempre "llegará tarde", sin aprender nada, obtiene una nota alta. Con el área bajo la curva ROC, ese modelo solo saca 0.5.


### 8.1 Clasificador SGD (descenso de gradiente estocástico)


In [10]:
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import cross_val_score

sgd_clf = make_pipeline(
    preprocessing,
    SGDClassifier(random_state=42)
)

sgd_scores = cross_val_score(
    sgd_clf,
    df_train_limpieza,
    df_train_labels,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

print("AUC ROC por partición:", sgd_scores.round(4))
print(f"AUC ROC promedio: {sgd_scores.mean():.4f}")

AUC ROC por partición: [0.7311 0.7249 0.7154 0.723  0.729 ]
AUC ROC promedio: 0.7247


### 8.2 Regresión logística


In [11]:
from sklearn.linear_model import LogisticRegression

logistica_clf = make_pipeline(
    preprocessing,
    LogisticRegression(random_state=42)
)

logistica_scores = cross_val_score(
    logistica_clf,
    df_train_limpieza,
    df_train_labels,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

print("AUC ROC por partición:", logistica_scores.round(4))
print(f"AUC ROC promedio: {logistica_scores.mean():.4f}")

AUC ROC por partición: [0.7427 0.7464 0.7352 0.7375 0.7396]
AUC ROC promedio: 0.7403


### 8.3 Árbol de decisión


In [12]:
from sklearn.tree import DecisionTreeClassifier

arbol_clf = make_pipeline(
    preprocessing,
    DecisionTreeClassifier(random_state=42)
)

arbol_scores = cross_val_score(
    arbol_clf,
    df_train_limpieza,
    df_train_labels,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

print("AUC ROC por partición:", arbol_scores.round(4))
print(f"AUC ROC promedio: {arbol_scores.mean():.4f}")

AUC ROC por partición: [0.6929 0.7042 0.6977 0.6779 0.6919]
AUC ROC promedio: 0.6929


### 8.4 Random Forest (bosque aleatorio)


In [13]:
from sklearn.ensemble import RandomForestClassifier

forest_clf = make_pipeline(
    preprocessing,
    RandomForestClassifier(random_state=42)
)

forest_scores = cross_val_score(
    forest_clf,
    df_train_limpieza,
    df_train_labels,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

print("AUC ROC por partición:", forest_scores.round(4))
print(f"AUC ROC promedio: {forest_scores.mean():.4f}")

AUC ROC por partición: [0.7973 0.8014 0.7902 0.7952 0.8014]
AUC ROC promedio: 0.7971


### 8.5 Comparación de resultados


In [14]:
comparacion_modelos = pd.DataFrame({
    "Modelo": ["SGDClassifier", "LogisticRegression", "DecisionTreeClassifier", "RandomForestClassifier"],
    "AUC ROC promedio": [sgd_scores.mean(), logistica_scores.mean(), arbol_scores.mean(), forest_scores.mean()],
    "Desviación estándar": [sgd_scores.std(), logistica_scores.std(), arbol_scores.std(), forest_scores.std()]
}).sort_values("AUC ROC promedio", ascending=False).reset_index(drop=True)

display(comparacion_modelos.round(4))

,Modelo,AUC ROC promedio,Desviación estándar
0,RandomForestClassifier,0.7971,0.0042
1,LogisticRegression,0.7403,0.0039
2,SGDClassifier,0.7247,0.0054
3,DecisionTreeClassifier,0.6929,0.0087


### 8.6 Interpretación de los resultados

Random Forest obtuvo la mayor AUC ROC (0.7971), seguido de la regresión logística (0.7403), SGD (0.7247) y el árbol de decisión (0.6929). Los cuatro superan claramente el valor de 0.5, que corresponde a un modelo que responde al azar.

La desviación estándar de Random Forest es de solo 0.0042, por lo que su resultado es estable entre particiones. Por esto se selecciona Random Forest para el afinamiento de hiperparámetros.


## 9. Afinamiento de hiperparámetros

En el cuaderno de trabajo se compararon dos métodos para afinar el bosque aleatorio. Grid Search obtuvo el mayor AUC ROC promedio en validación cruzada (0.8360), seguido de Randomized Search (0.8224). En este cuaderno se reproduce la búsqueda ganadora.

Grid Search evalúa 16 combinaciones de cuatro hiperparámetros: el número de grupos geográficos (`n_clusters`), la cantidad de características consideradas en cada división (`max_features`), la profundidad máxima de los árboles (`max_depth`) y el mínimo de registros por hoja (`min_samples_leaf`). Cada combinación se evalúa con tres particiones del conjunto de entrenamiento.


### 9.1 Búsqueda en cuadrícula (Grid Search)


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("random_forest", RandomForestClassifier(random_state=42))
])

param_grid = {
    "preprocessing__geo__clustersimilarity__n_clusters": [20, 45],
    "random_forest__max_features": [10, 16],
    "random_forest__max_depth": [20, None],
    "random_forest__min_samples_leaf": [1, 3]
}

grid_search = GridSearchCV(
    full_pipeline,
    param_grid=param_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1
)

grid_search.fit(df_train_limpieza, df_train_labels)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'preprocessing__geo__cl...rsimilarity__n_clusters': [20, 45], 'random_forest__max_depth': [20, None], 'random_forest__max_features': [10, 16], 'random_forest__min_samples_leaf': [1, 3]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score

### 9.2 Resultados de la búsqueda


In [17]:
cv_res = pd.DataFrame(grid_search.cv_results_)
cv_res = cv_res.sort_values("mean_test_score", ascending=False)

cv_res = cv_res[[
    "param_preprocessing__geo__clustersimilarity__n_clusters",
    "param_random_forest__max_features",
    "param_random_forest__max_depth",
    "param_random_forest__min_samples_leaf",
    "split0_test_score",
    "split1_test_score",
    "split2_test_score",
    "mean_test_score"
]]

cv_res.columns = [
    "n_clusters", "max_features", "max_depth", "min_samples_leaf",
    "AUC partición 1", "AUC partición 2", "AUC partición 3", "AUC promedio"
]

modelo_final = grid_search.best_estimator_

print("Configuraciones probadas:", len(cv_res))
print("Mejor configuración:", grid_search.best_params_)
print(f"Mejor AUC ROC promedio: {grid_search.best_score_:.4f}")

display(cv_res.round(4).reset_index(drop=True))

Configuraciones probadas: 16
Mejor configuración: {'preprocessing__geo__clustersimilarity__n_clusters': 45, 'random_forest__max_depth': None, 'random_forest__max_features': 10, 'random_forest__min_samples_leaf': 1}
Mejor AUC ROC promedio: 0.8360


,n_clusters,max_features,max_depth,min_samples_leaf,AUC partición 1,AUC partición 2,AUC partición 3,AUC promedio
0,45,10,None,1,0.8393,0.8331,0.8357,0.8360
1,45,16,None,1,0.8353,0.8268,0.8297,0.8306
2,45,10,None,3,0.8290,0.8228,0.8239,0.8252
3,45,16,None,3,0.8266,0.8185,0.8202,0.8218
4,45,10,20,1,0.8193,0.8092,0.8166,0.8150
5,20,10,None,1,0.8173,0.8126,0.8128,0.8142
6,20,16,None,1,0.8161,0.8120,0.8122,0.8134
7,45,16,20,1,0.8194,0.8081,0.8115,0.8130
8,45,10,20,3,0.8132,0.8042,0.8071,0.8082
9,45,16,20,3,0.8146,0.8024,0.8053,0.8074


### 9.3 Interpretación del afinamiento

De las 16 configuraciones evaluadas con Grid Search, la que obtuvo el mayor AUC ROC promedio (0.8360) utilizó 45 grupos geográficos. El bosque consideró 10 características en cada división, sin un límite fijo de profundidad y con un mínimo de un registro por hoja.

Se seleccionó esta configuración para evaluar el modelo con los datos de prueba reservados.


## 10. Evaluación final con el conjunto de prueba


In [18]:
from sklearn.metrics import roc_auc_score

probabilidades_test = modelo_final.predict_proba(X_test_estratificado)[:, 1]
auc_test = roc_auc_score(y_test_estratificado, probabilidades_test)

print(f"Registros de prueba: {len(y_test_estratificado):,}")
print(f"AUC ROC en prueba: {auc_test:.4f}")

Registros de prueba: 36,104
AUC ROC en prueba: 0.8804


### 10.1 Intervalo de confianza del 95 % para el AUC ROC


In [19]:
import numpy as np

y_test = y_test_estratificado.to_numpy()
indices_0 = np.flatnonzero(y_test == 0)
indices_1 = np.flatnonzero(y_test == 1)

rng = np.random.default_rng(42)
auc_bootstrap = []

for _ in range(2000):
    muestra_0 = rng.choice(indices_0, size=len(indices_0), replace=True)
    muestra_1 = rng.choice(indices_1, size=len(indices_1), replace=True)
    indices = np.concatenate([muestra_0, muestra_1])

    auc = roc_auc_score(y_test[indices], probabilidades_test[indices])
    auc_bootstrap.append(auc)

limite_inferior, limite_superior = np.percentile(auc_bootstrap, [2.5, 97.5])

print(f"AUC ROC en prueba: {auc_test:.4f}")
print(f"IC del 95 %: [{limite_inferior:.4f}, {limite_superior:.4f}]")

AUC ROC en prueba: 0.8804
IC del 95 %: [0.8770, 0.8838]


### 10.2 Importancia de las características del modelo final


In [20]:
ranking_caracteristicas = pd.DataFrame({
    "Característica": modelo_final["preprocessing"].get_feature_names_out(),
    "Importancia": modelo_final["random_forest"].feature_importances_
})

ranking_caracteristicas = ranking_caracteristicas.sort_values(
    "Importancia", ascending=False, ignore_index=True
)

display(ranking_caracteristicas.head(10).round(4))

,Característica,Importancia
0,cat__Shipping Mode_Standard Class,0.0782
1,num__Days for shipment (scheduled),0.0633
2,cat__Shipping Mode_First Class,0.0434
3,profit_ratio__ratio,0.0262
4,num__Benefit per order,0.0260
5,num__Order Item Profit Ratio,0.0238
6,num__Order Item Total,0.0206
7,log__Order Item Total,0.0205
8,log_revenue_ratio__ratio,0.0197
9,revenue_ratio__ratio,0.0197


### 10.3 Conclusión

El bosque aleatorio obtuvo un AUC ROC de **0.8804** en los **36 104 registros de prueba**, con un intervalo de confianza del 95 % entre **0.8770 y 0.8838**. Este resultado indica que el modelo suele asignar mayor riesgo a una entrega tardía que a una entrega a tiempo.

Las tres características con mayor peso fueron la modalidad de envío Standard Class (7.82 %), los días programados para el envío (6.33 %) y la modalidad First Class (4.34 %). En conjunto, representaron el **18.49 %** de la importancia total del modelo.
